# LogReg Toxic Comment Classifier: Binary & Multi-Label

Recreates the `baseline.py` LogisticRegression pipeline in notebook form for both:
- **Binary** (`is_toxic`): any of 6 labels set → 1
- **Multi-label**: 6 toxicity categories simultaneously

**Features**: word TF-IDF (100k, unigrams+bigrams) · `ALL_STOPWORDS` (sklearn English + Wikipedia domain) · `class_weight='balanced'` · per-label threshold tuning on validation set

**Checklist**:
1. Compare binary vs multi-label pipeline quality (§6)
2. Define evaluation metrics with moderation context (§3)
3. Training data size ablation — dataset combination proxy (§7)
4. Label quality investigation — FP/FN error analysis (§8)
5. Word-level feature rationale (§2)
6. Two-model architecture narrative (§9)

In [ ]:
import sys, warnings
sys.path.insert(0, '../models')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    precision_recall_fscore_support, roc_curve, precision_recall_curve,
    confusion_matrix,
)

from baseline import ALL_STOPWORDS, RANDOM_STATE, LABELS, RARE_LABELS
from preprocessing import preprocess_df

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

THRESHOLDS_GRID = np.arange(0.1, 0.9, 0.02)
print('Imports OK')

In [ ]:
CONFIG = {
    'paths': {
        'train':       '../data/train_cleaned.parquet',
        'test':        '../data/test_cleaned.parquet',
        'test_orig':   '../data/test.parquet',          # original — has all 153k IDs + comment_text
        'test_labels': '../data/test_labels_cleaned.parquet',
    },
    'model': {
        'C':            0.5,
        'solver':       'liblinear',
        'class_weight': 'balanced',
        'max_iter':     1000,
    },
    'val_size':    0.1,
    'random_state': RANDOM_STATE,
}

In [ ]:
train_raw   = pd.read_parquet(CONFIG['paths']['train'])
test_raw    = pd.read_parquet(CONFIG['paths']['test'])
test_labels = pd.read_parquet(CONFIG['paths']['test_labels'])

# Filter withheld rows (toxic == -1) — already removed in cleaned files, but guard just in case
if 'toxic' in test_labels.columns and (test_labels['toxic'] == -1).any():
    evaluable = test_labels[test_labels['toxic'] != -1].copy().reset_index(drop=True)
else:
    evaluable = test_labels.copy().reset_index(drop=True)

# Always pull comment_text from the original test file (guaranteed to have every ID).
# Drop it from evaluable first to avoid _x/_y column conflicts on merge.
test_text = pd.read_parquet(CONFIG['paths']['test_orig'])[['id', 'comment_text']]
eval_df = (
    evaluable
    .drop(columns=['comment_text'], errors='ignore')
    .merge(test_text, on='id', how='left')
    .reset_index(drop=True)
)

# Binary target: is any of the 6 labels set?
train_raw['is_toxic'] = (train_raw[LABELS].sum(axis=1) > 0).astype(int)
eval_df['is_toxic']   = (eval_df[LABELS].sum(axis=1) > 0).astype(int)

print(f"Train  : {len(train_raw):,} rows | toxic {train_raw['is_toxic'].sum():,} ({train_raw['is_toxic'].mean()*100:.1f}%)")
print(f"Eval   : {len(eval_df):,} rows  | toxic {eval_df['is_toxic'].sum():,} ({eval_df['is_toxic'].mean()*100:.1f}%)")
print()
print('Multi-label positive rates (train):')
for label in LABELS:
    marker = ' *rare*' if label in RARE_LABELS else ''
    print(f'  {label:<20} {train_raw[label].mean()*100:.2f}%{marker}')

In [ ]:
rs = CONFIG['random_state']
m  = CONFIG['model']

train_fit, train_val = train_test_split(
    train_raw, test_size=CONFIG['val_size'], random_state=rs, stratify=train_raw['toxic']
)
print(f'Train-fit : {len(train_fit):,}  |  Val (threshold tuning): {len(train_val):,}')

print('\nPreprocessing text (lowercasing, URL/HTML stripping, repeated-char collapsing)...')
X_fit_text  = preprocess_df(train_fit['comment_text'])
X_val_text  = preprocess_df(train_val['comment_text'])
X_test_text = preprocess_df(eval_df['comment_text'])

# Binary targets
y_fit_bin  = train_fit['is_toxic'].values
y_val_bin  = train_val['is_toxic'].values
y_test_bin = eval_df['is_toxic'].values

# Multi-label targets (n, 6)
y_fit_multi  = train_fit[LABELS].values
y_val_multi  = train_val[LABELS].values
y_test_multi = eval_df[LABELS].values

In [ ]:
print(f'Fitting word TF-IDF — stop words: {len(ALL_STOPWORDS)} (sklearn English + domain artifacts)...')
tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=100_000,
    ngram_range=(1, 2),
    min_df=3,
    analyzer='word',
    stop_words=ALL_STOPWORDS,
)
X_fit_raw  = tfidf.fit_transform(X_fit_text)
X_val_raw  = tfidf.transform(X_val_text)
X_test_raw = tfidf.transform(X_test_text)

scaler = MaxAbsScaler()
X_fit  = scaler.fit_transform(X_fit_raw)
X_val  = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

feat_names = np.array(tfidf.get_feature_names_out())
print(f'Feature matrix: train={X_fit.shape}, val={X_val.shape}, test={X_test.shape}')

## 2. Feature Choice: Word TF-IDF Only

Char n-grams (3–5 chars, 20k features) are designed to catch obfuscated toxic words like `f*ck` or `a$$`. However:

- `clean_text()` already normalises repeated characters and strips punctuation noise
- The TF-IDF ablation in `imbalanced_binary_classification.ipynb` found word-only TF-IDF achieves **0.8705 ± 0.0021 macro F1** — the word+char combination showed no meaningful improvement over this
- Dropping char n-grams removes ~20k features and roughly halves fit time

**Decision**: use word TF-IDF only. Revisit if char TF-IDF shows >0.5pp macro F1 gain.

**Stop words** = `sklearn.ENGLISH_STOP_WORDS` ∪ `DOMAIN_STOPWORDS` (30 Wikipedia platform artifacts: `wikipedia`, `utc`, `talk`, `edit`, …). EDA confirmed these dominate the token distribution without carrying any toxicity signal.

## 3. Evaluation Metrics

For **content moderation**, the cost of errors is asymmetric:

| Metric | Formula | Moderation meaning |
|---|---|---|
| **Recall** (sensitivity) | TP / (TP+FN) | % of toxic comments caught — **primary safety metric** |
| **Precision** | TP / (TP+FP) | % of flagged comments truly toxic — affects user trust |
| **F1** | harmonic mean | Balanced tradeoff; macro-F1 for multi-label |
| **AUC-ROC** | area under ROC | Discrimination quality regardless of threshold |
| **Avg Precision** (AP) | area under PR | More informative than AUC under heavy imbalance |
| **Specificity** (TNR) | TN / (TN+FP) | % of clean comments correctly left unflagged |
| **Clean accuracy** | all-zero rows correct | Fraction of fully clean comments never flagged |

**Primary**: F1 (balanced, threshold-tuned)  
**Hierarchy**: Recall > Precision for safety-critical deployments (missed toxic content = user harm)  
**Threshold tuning**: grid search over [0.10, 0.88] in 0.02 steps on the validation set, maximising F1

In [ ]:
def find_best_threshold(y_true, y_proba):
    best_f1, best_t = 0.0, 0.5
    for t in THRESHOLDS_GRID:
        f = f1_score(y_true, (y_proba >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1

def find_best_thresholds_multi(y_true, y_proba):
    thresholds = np.full(len(LABELS), 0.5)
    for i in range(len(LABELS)):
        best_f1, best_t = 0.0, 0.5
        for t in THRESHOLDS_GRID:
            f = f1_score(y_true[:, i], (y_proba[:, i] >= t).astype(int), zero_division=0)
            if f > best_f1:
                best_f1, best_t = f, t
        thresholds[i] = best_t
    return thresholds

def binary_metrics(y_true, y_pred, y_proba):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {
        'precision': p, 'recall': r, 'f1': f,
        'auc':       roc_auc_score(y_true, y_proba),
        'ap':        average_precision_score(y_true, y_proba),
    }

print('Helper functions defined.')

---

## 4. Binary Pipeline: `is_toxic`

Collapses all 6 labels into a single binary target.
**Use case**: fast first-pass moderation gate — flag anything that might be toxic.

In [ ]:
print('Training binary LogisticRegression...')
bin_lr = LogisticRegression(
    C=m['C'], class_weight=m['class_weight'],
    solver=m['solver'], max_iter=m['max_iter'], random_state=rs,
)
bin_lr.fit(X_fit, y_fit_bin)

# Threshold tuning on held-out val set
val_proba_bin             = bin_lr.predict_proba(X_val)[:, 1]
best_thresh_bin, best_val_f1 = find_best_threshold(y_val_bin, val_proba_bin)
print(f'Val best threshold: {best_thresh_bin:.2f}  (val F1 = {best_val_f1:.4f})')

# Test evaluation
test_proba_bin = bin_lr.predict_proba(X_test)[:, 1]
test_pred_bin  = (test_proba_bin >= best_thresh_bin).astype(int)
bin_m          = binary_metrics(y_test_bin, test_pred_bin, test_proba_bin)

print(f"\n{'='*50}")
print(f'  Binary LR  [threshold={best_thresh_bin:.2f}]')
print(f"{'='*50}")
for k, v in bin_m.items():
    print(f'  {k:<12}: {v:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Confusion matrix
cm = confusion_matrix(y_test_bin, test_pred_bin)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: clean', 'Pred: toxic'],
            yticklabels=['True: clean', 'True: toxic'])
axes[0].set_title(f'Confusion Matrix [thresh={best_thresh_bin:.2f}]')

# ROC curve
fpr, tpr, _ = roc_curve(y_test_bin, test_proba_bin)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC={bin_m["auc"]:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()

# PR curve
prec_c, rec_c, _ = precision_recall_curve(y_test_bin, test_proba_bin)
axes[2].plot(rec_c, prec_c, lw=2, label=f'AP={bin_m["ap"]:.4f}')
axes[2].axhline(y_test_bin.mean(), color='grey', ls='--', lw=1, label='baseline rate')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision–Recall Curve'); axes[2].legend()

plt.suptitle('Binary LR — Toxicity Detection', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
coef  = bin_lr.coef_[0]
top_n = 20
top_toxic    = np.argsort(coef)[-top_n:][::-1]
top_nontoxic = np.argsort(coef)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, idx, title, color in [
    (axes[0], top_toxic,    f'Top {top_n} → Toxic features',     'tomato'),
    (axes[1], top_nontoxic, f'Top {top_n} → Non-toxic features', 'steelblue'),
]:
    ax.barh(feat_names[idx][::-1], coef[idx][::-1], color=color)
    ax.set_xlabel('LR coefficient')
    ax.set_title(title)
plt.suptitle('Binary LR — Feature Weights', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
ann = eval_df[['comment_text', 'is_toxic']].copy()
ann['proba'] = test_proba_bin
ann['pred']  = test_pred_bin

fn = ann[(ann['is_toxic'] == 1) & (ann['pred'] == 0)]
fp = ann[(ann['is_toxic'] == 0) & (ann['pred'] == 1)]

print(f'False Negatives (missed toxic): {len(fn):,}  miss-rate={len(fn)/y_test_bin.sum():.1%}')
print(fn.nsmallest(5, 'proba')[['comment_text', 'proba']].to_string(index=False))

print(f'\nFalse Positives (false alarms): {len(fp):,}  FPR={len(fp)/(y_test_bin==0).sum():.1%}')
print(fp.nlargest(5, 'proba')[['comment_text', 'proba']].to_string(index=False))

---

## 5. Multi-Label Pipeline: 6 Toxicity Categories

Predicts all 6 labels simultaneously with `MultiOutputClassifier(LogisticRegression)`.  
Each label gets its own LR estimator with an independently tuned threshold.

**Use case**: detailed categorisation for routing — threats escalate differently from insults.

In [ ]:
print('Training multi-label LogisticRegression (6 binary estimators)...')
multi_lr = MultiOutputClassifier(
    LogisticRegression(
        C=m['C'], class_weight=m['class_weight'],
        solver=m['solver'], max_iter=m['max_iter'], random_state=rs,
    ),
    n_jobs=-1,
)
multi_lr.fit(X_fit, y_fit_multi)

# Per-label threshold tuning on val set
val_proba_list  = multi_lr.predict_proba(X_val)
val_proba_multi = np.column_stack([p[:, 1] for p in val_proba_list])
thresholds_multi = find_best_thresholds_multi(y_val_multi, val_proba_multi)

print('Per-label thresholds:')
for label, t in zip(LABELS, thresholds_multi):
    marker = '  ← rare (<0.5% positive rate)' if label in RARE_LABELS else ''
    print(f'  {label:<20} {t:.2f}{marker}')

In [ ]:
test_proba_list  = multi_lr.predict_proba(X_test)
test_proba_multi = np.column_stack([p[:, 1] for p in test_proba_list])
test_pred_multi  = (test_proba_multi >= thresholds_multi).astype(int)

rows = []
for i, label in enumerate(LABELS):
    p, r, f, _ = precision_recall_fscore_support(
        y_test_multi[:, i], test_pred_multi[:, i], average='binary', zero_division=0
    )
    auc = roc_auc_score(y_test_multi[:, i], test_proba_multi[:, i]) if y_test_multi[:, i].sum() > 0 else float('nan')
    rows.append({'Label': label, 'Precision': round(p,4), 'Recall': round(r,4),
                 'F1': round(f,4), 'AUC': round(auc,4), 'Threshold': round(thresholds_multi[i],2)})

metrics_df = pd.DataFrame(rows).set_index('Label')
macro_f1   = f1_score(y_test_multi, test_pred_multi, average='macro', zero_division=0)
micro_f1   = f1_score(y_test_multi, test_pred_multi, average='micro', zero_division=0)
macro_auc  = float(np.nanmean(metrics_df['AUC'].values))

clean_mask = y_test_multi.sum(axis=1) == 0
clean_acc  = (test_pred_multi[clean_mask].sum(axis=1) == 0).sum() / clean_mask.sum()

print(f"{'='*65}")
print(f'  Multi-Label LR  [per-label tuned thresholds]')
print(f"{'='*65}")
print(metrics_df.to_string())
print(f"{'─'*65}")
print(f'  Macro F1  : {macro_f1:.4f}   Micro F1 : {micro_f1:.4f}   Macro AUC : {macro_auc:.4f}')
print(f'  Clean acc : {clean_acc:.4f}  (all-zero rows correctly left unflagged)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Per-label F1
bars = axes[0].bar(metrics_df.index, metrics_df['F1'],
                   color=sns.color_palette('muted', len(LABELS)))
axes[0].axhline(macro_f1, color='red', ls='--', lw=1.5, label=f'macro F1={macro_f1:.4f}')
for bar, f in zip(bars, metrics_df['F1']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{f:.3f}', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('F1')
axes[0].set_title('Per-Label F1')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

# Precision vs Recall per label
x = np.arange(len(LABELS))
w = 0.35
axes[1].bar(x - w/2, metrics_df['Precision'], w, label='Precision', color='steelblue')
axes[1].bar(x + w/2, metrics_df['Recall'],    w, label='Recall',    color='tomato')
axes[1].set_xticks(x)
axes[1].set_xticklabels(LABELS, rotation=30, ha='right')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision vs Recall per Label')
axes[1].legend()

plt.suptitle('Multi-Label LR — Per-Label Metrics', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
top_n = 15
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, estimator, label in zip(axes.ravel(), multi_lr.estimators_, LABELS):
    coef = estimator.coef_[0]
    idx  = np.argsort(coef)[-top_n:][::-1]
    ax.barh(feat_names[idx][::-1], coef[idx][::-1], color='tomato')
    ax.set_title(f'Top features — {label}', fontsize=10)
    ax.set_xlabel('LR coefficient')
plt.suptitle('Multi-Label LR — Per-Label Top Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
error_rows = []
for i, label in enumerate(LABELS):
    tn = int(((y_test_multi[:, i] == 0) & (test_pred_multi[:, i] == 0)).sum())
    fp = int(((y_test_multi[:, i] == 0) & (test_pred_multi[:, i] == 1)).sum())
    fn = int(((y_test_multi[:, i] == 1) & (test_pred_multi[:, i] == 0)).sum())
    tp = int(((y_test_multi[:, i] == 1) & (test_pred_multi[:, i] == 1)).sum())
    fpr_v = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr_v = fn / (fn + tp) if (fn + tp) > 0 else 0
    error_rows.append({'Label': label, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                       'FPR': round(fpr_v, 3), 'FNR (miss rate)': round(fnr_v, 3)})

error_df = pd.DataFrame(error_rows).set_index('Label')
print('Per-label error breakdown:')
print(error_df.to_string())

# Top 5 false negatives for toxic label (hardest misses)
toxic_i  = LABELS.index('toxic')
mask_fn  = (y_test_multi[:, toxic_i] == 1) & (test_pred_multi[:, toxic_i] == 0)
fn_toxic = eval_df[mask_fn][['comment_text']].copy()
fn_toxic['proba'] = test_proba_multi[mask_fn, toxic_i].round(3)
print(f'\nTop 5 false negatives — toxic label (lowest confidence):')
print(fn_toxic.nsmallest(5, 'proba')[['comment_text', 'proba']].to_string(index=False))

---

## 6. Pipeline Comparison: Binary vs Multi-Label

Both models use identical features and LR hyperparameters.  
To compare fairly on the binary detection task, we collapse multi-label predictions: if any of the 6 labels is predicted → `is_toxic = 1`.

In [ ]:
# Collapse multi-label predictions to binary
multi_pred_collapsed = (test_pred_multi.sum(axis=1) > 0).astype(int)
multi_proba_max      = test_proba_multi.max(axis=1)

p_mc, r_mc, f_mc, _ = precision_recall_fscore_support(
    y_test_bin, multi_pred_collapsed, average='binary', zero_division=0
)
auc_mc = roc_auc_score(y_test_bin, multi_proba_max)
ap_mc  = average_precision_score(y_test_bin, multi_proba_max)

comp = pd.DataFrame({
    'Model': ['Binary LR', 'Multi-Label LR (collapsed)'],
    'Precision': [round(bin_m['precision'], 4), round(p_mc, 4)],
    'Recall':    [round(bin_m['recall'],    4), round(r_mc, 4)],
    'F1':        [round(bin_m['f1'],        4), round(f_mc, 4)],
    'AUC-ROC':   [round(bin_m['auc'],       4), round(auc_mc, 4)],
    'Avg Prec':  [round(bin_m['ap'],        4), round(ap_mc,  4)],
}).set_index('Model')

print('Binary detection quality — head-to-head:')
print(comp.to_string())
print()
print('Interpretation:')
print('  Binary LR is trained directly on is_toxic → maximises binary recall')
print('  Multi-label collapsed: each label threshold independently tuned → lower binary recall')
print('  For a gating pass, prefer the binary model; for content categorisation, use multi-label')

---

## 7. Training Data Size Ablation (Dataset Combination Proxy)

**Question**: how much does adding more labeled training data improve recall and precision?  
We train on 25 / 50 / 75 / 100 % of the training set and evaluate on the held-out test.

This acts as a **proxy for dataset combination** (adding external datasets like WoT / Dota / CivilComments):
- If recall still grows steeply at 100%, external datasets would provide meaningful additional lift
- If recall plateaus, the model is near saturation at the current feature capacity
- A growing **precision gap** at small data sizes tells you where annotation noise dominates

In [ ]:
fracs     = [0.25, 0.50, 0.75, 1.00]
size_rows = []
rng       = np.random.default_rng(rs)

n_fit = X_fit.shape[0]   # sparse matrix: use .shape[0] instead of len()

for frac in fracs:
    n = int(n_fit * frac)
    if frac < 1.0:
        idx    = rng.choice(n_fit, size=n, replace=False)
        X_sub  = X_fit[idx]
        yb_sub = y_fit_bin[idx]
    else:
        X_sub, yb_sub = X_fit, y_fit_bin

    lr_tmp = LogisticRegression(C=m['C'], class_weight='balanced',
                                solver='liblinear', max_iter=1000, random_state=rs)
    lr_tmp.fit(X_sub, yb_sub)
    vp  = lr_tmp.predict_proba(X_val)[:, 1]
    t,_ = find_best_threshold(y_val_bin, vp)
    tp  = lr_tmp.predict_proba(X_test)[:, 1]
    pred = (tp >= t).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_test_bin, pred, average='binary', zero_division=0)
    size_rows.append({
        'Train fraction': f'{frac:.0%}', 'N_train': n,
        'Precision': round(p, 4), 'Recall': round(r, 4), 'F1': round(f, 4), 'Threshold': round(t, 2)
    })
    print(f'{frac:.0%}  ({n:,} rows)  P={p:.4f}  R={r:.4f}  F1={f:.4f}  thresh={t:.2f}')

size_df = pd.DataFrame(size_rows).set_index('Train fraction')
print()
print(size_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in [
    (axes[0], 'Precision', 'steelblue'),
    (axes[1], 'Recall',    'tomato'),
    (axes[2], 'F1',        'green'),
]:
    vals = size_df[col].values
    ax.plot(fracs, vals, marker='o', color=color, lw=2)
    for fx, vy in zip(fracs, vals):
        ax.annotate(f'{vy:.4f}', (fx, vy), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=9)
    ax.set_xticks(fracs)
    ax.set_xticklabels([f'{f:.0%}' for f in fracs])
    ax.set_xlabel('Training set fraction')
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.set_ylim(min(vals) - 0.05, max(vals) + 0.05)

plt.suptitle('Training Data Size vs Binary Detection Quality', fontsize=13)
plt.tight_layout()
plt.show()

---

## 8. Label Quality Investigation

Not all model errors are model failures — some are **annotation inconsistencies**:

- **High-confidence FP** (proba > 0.80, labeled non-toxic): The model is very sure this is toxic but the label disagrees. These are prime candidates for annotation review — the comment may genuinely be toxic but was missed by annotators.
- **High-confidence FN** (proba < 0.15, labeled toxic): The model sees nothing toxic but the label says otherwise. May indicate context-dependent toxicity, sarcasm, or dog-whistling that surface-form TF-IDF cannot capture.
- **Borderline zone** (0.40–0.60 proba): Comments where the model is genuinely uncertain — human annotators are likely to disagree here too.

In [ ]:
ann = eval_df[['comment_text', 'is_toxic']].copy()
ann['proba'] = test_proba_bin
ann['pred']  = test_pred_bin

hc_fp = ann[(ann['is_toxic'] == 0) & (ann['proba'] > 0.80)]
hc_fn = ann[(ann['is_toxic'] == 1) & (ann['proba'] < 0.15)]
borderline = ann[ann['proba'].between(0.40, 0.60)]

print(f'High-confidence FP (proba>0.80, labeled non-toxic): {len(hc_fp):,}')
print('Potential annotation misses — review candidates:')
print(hc_fp.nlargest(5, 'proba')[['comment_text', 'proba']].to_string(index=False))

print(f'\nHigh-confidence FN (proba<0.15, labeled toxic): {len(hc_fn):,}')
print('Context-dependent or surface-benign content:')
print(hc_fn.nsmallest(5, 'proba')[['comment_text', 'proba']].to_string(index=False))

borderline_acc = (borderline['is_toxic'] == (borderline['proba'] >= 0.5).astype(int)).mean()
print(f'\nBorderline zone (0.40–0.60 proba): {len(borderline):,} comments')
print(f'  Model accuracy in this zone: {borderline_acc:.1%}')
print('  Annotator agreement in borderline zones is typically <80% — these are genuinely ambiguous')

In [ ]:
# For each high-confidence FP, which multi-label category does the model assign?
# Helps diagnose which type of toxicity the model thinks it sees vs what was annotated.
hc_fp_idx   = hc_fp.nlargest(20, 'proba').index.tolist()
hc_fp_proba = test_proba_multi[hc_fp_idx]

print('Top 5 high-confidence FP — multi-label probabilities (which category?)')
for pos, (row_i, proba_row) in enumerate(zip(hc_fp_idx[:5], hc_fp_proba[:5])):
    text = eval_df.loc[row_i, 'comment_text']
    print(f'\n[{pos+1}] text: {text[:120]}...' if len(text) > 120 else f'\n[{pos+1}] text: {text}')
    active = [(LABELS[j], p) for j, p in enumerate(proba_row) if p > 0.25]
    print('  Model flags:', ', '.join(f'{lbl}={p:.2f}' for lbl, p in active) if active else 'none > 0.25')

# Annotation consistency proxy: how often does multi-label agree with binary on FP cases
# (both can be wrong, but disagreement = model-annotation mismatch, not just binary FP)
binary_say_toxic = ann['pred'] == 1
multi_say_toxic  = (test_pred_multi.sum(axis=1) > 0)
both_say_toxic   = binary_say_toxic & multi_say_toxic
labeled_nontoxic = eval_df['is_toxic'] == 0

both_flag_clean = (both_say_toxic & labeled_nontoxic).sum()
print(f'\nRows flagged by BOTH models but labeled non-toxic: {both_flag_clean:,}')
print('These are the strongest candidates for annotation review.')

---

## 9. Two-Model Architecture Narrative

### Why two models are better than one

#### Stage 1 — Binary LR (fast, high-recall gating)
- Trained directly on `is_toxic` → threshold tuned to maximise F1, biased toward recall
- Runs on every comment at publish time
- Goal: catch all toxic content before it reaches users; a miss (FN) is more costly than a false alarm (FP)
- Handles the 90% of traffic that is clearly clean in a single forward pass

#### Stage 2 — Multi-Label LR (detailed categorisation)
- Runs only on comments that passed Stage 1 → processes ~10% of total traffic
- Each label routes to a different action:
  - `threat` → escalate to law enforcement pipeline
  - `identity_hate` → high-priority human review queue
  - `severe_toxic` → immediate block + repeat-offender flag
  - `obscene` / `insult` → automated warning or content filter
- Per-label threshold tuning lets each category operate at its own precision/recall tradeoff

#### Performance tradeoff (from §6)
| Model | Precision | Recall | F1 |
|---|---|---|---|
| Binary LR | higher | higher | higher for binary task |
| Multi-label (collapsed) | lower | lower | lower for binary task |

The binary model wins on the gating task because it is directly optimised for it.  
The multi-label model wins on categorisation because its 6 per-label thresholds are independently tuned.

#### When one model suffices
- Binary only: simple filter, no downstream routing, compute budget tight
- Multi-label only: if `toxic` label recall is comparable to binary (check §6 metrics), a single model reduces deployment complexity

#### Training data combination (from §7)
The size ablation tells us whether external datasets (WoT, Dota, CivilComments) would help:
- If Recall still grows steeply at 100% of Jigsaw training data → **yes, add more data**
- If Recall plateaus → model is feature-saturated; better features (e.g. BERT embeddings) are needed rather than more bag-of-words data

#### Label quality impact (from §8)
High-confidence FPs flagged by both models but labeled non-toxic are the most actionable for re-annotation. Including corrected labels in retraining tends to improve precision without hurting recall — the model is already detecting these; it just needs the ground truth to confirm it.